# Notebook 05: Hashes — Mini Objects in Redis

Hashes are **maps of field-value pairs** stored under a single key. Think of them like a **Python dictionary** inside Redis.

Instead of this (separate keys):
```
user:1001:name  → "Sujit"
user:1001:email → "sujit@example.com"
user:1001:age   → "25"
```

You store this (one hash):
```
user:1001 → { name: "Sujit", email: "sujit@example.com", age: "25" }
```

**Benefits:** More memory-efficient, logically grouped, and you can read/write individual fields.

**Common uses:** User profiles, product details, shopping carts, session data, configuration.

In [ ]:
import redis
import json

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## 1. HSET — Set Fields in a Hash

In [ ]:
# Set a single field
# Redis CLI: HSET user:1001 name "Sujit"
r.hset('user:1001', 'name', 'Sujit')

# Set multiple fields at once using mapping
# Redis CLI: HSET user:1001 email "sujit@example.com" age 25 city "Mumbai"
r.hset('user:1001', mapping={
    'email': 'sujit@example.com',
    'age': 25,
    'city': 'Mumbai',
    'role': 'developer'
})

print("User hash created!")

## 2. HGET and HGETALL — Reading Hash Data

In [ ]:
# HGET — Get a single field
# Redis CLI: HGET user:1001 name
name = r.hget('user:1001', 'name')
print(f"Name: {name}")

# HGETALL — Get ALL fields and values as a dictionary
# Redis CLI: HGETALL user:1001
user = r.hgetall('user:1001')
print(f"\nFull user profile:")
for field, value in user.items():
    print(f"  {field}: {value}")

print(f"\nReturned type: {type(user)}")  # It's a Python dict!

In [ ]:
# HMGET — Get multiple specific fields
# Redis CLI: HMGET user:1001 name email
values = r.hmget('user:1001', 'name', 'email', 'phone')  # 'phone' doesn't exist
print(f"name, email, phone: {values}")
# Note: non-existent field returns None

## 3. HEXISTS, HDEL, HLEN

In [ ]:
# HEXISTS — Check if a field exists
# Redis CLI: HEXISTS user:1001 email
print(f"Has email? {r.hexists('user:1001', 'email')}")   # True
print(f"Has phone? {r.hexists('user:1001', 'phone')}")   # False

# HLEN — Number of fields in the hash
# Redis CLI: HLEN user:1001
print(f"Number of fields: {r.hlen('user:1001')}")

# HDEL — Delete a field
# Redis CLI: HDEL user:1001 city
r.hdel('user:1001', 'city')
print(f"After deleting 'city': {r.hgetall('user:1001')}")

## 4. HKEYS and HVALS

In [ ]:
# HKEYS — Get all field names
# Redis CLI: HKEYS user:1001
print(f"Fields: {r.hkeys('user:1001')}")

# HVALS — Get all values
# Redis CLI: HVALS user:1001
print(f"Values: {r.hvals('user:1001')}")

## 5. HINCRBY / HINCRBYFLOAT — Increment Hash Fields

In [ ]:
# Numeric fields can be incremented atomically!
print(f"Age before: {r.hget('user:1001', 'age')}")

# Redis CLI: HINCRBY user:1001 age 1
r.hincrby('user:1001', 'age', 1)  # Happy birthday!
print(f"Age after +1: {r.hget('user:1001', 'age')}")

# Works with float too
r.hset('product:1', mapping={'name': 'Widget', 'price': '9.99'})
# Redis CLI: HINCRBYFLOAT product:1 price 2.50
r.hincrbyfloat('product:1', 'price', 2.50)
print(f"Price after +2.50: ${r.hget('product:1', 'price')}")

## 6. HSETNX — Set Only If Field Doesn't Exist

In [ ]:
# Redis CLI: HSETNX user:1001 name "Someone Else"
result1 = r.hsetnx('user:1001', 'name', 'Someone Else')  # Fails — name exists
print(f"Set existing 'name': {result1} (0 = not set)")
print(f"Name is still: {r.hget('user:1001', 'name')}")

result2 = r.hsetnx('user:1001', 'phone', '123-456-7890')  # Works — phone is new
print(f"Set new 'phone': {result2} (1 = set)")
print(f"Phone: {r.hget('user:1001', 'phone')}")

---
## 7. Hash vs Separate Keys — Why Hashes Win

Hashes with fewer than ~128 fields use a compact **ziplist** encoding internally, which uses much less memory than separate string keys.

In [ ]:
r.flushdb()

# Approach 1: Separate keys (inefficient)
for i in range(100):
    r.set(f'user:{i}:name', f'User_{i}')
    r.set(f'user:{i}:email', f'user{i}@test.com')

mem_separate = r.info('memory')['used_memory']
keys_separate = r.dbsize()
print(f"Separate keys: {keys_separate} keys, {mem_separate:,} bytes")

r.flushdb()

# Approach 2: Hashes (efficient)
for i in range(100):
    r.hset(f'user:{i}', mapping={'name': f'User_{i}', 'email': f'user{i}@test.com'})

mem_hash = r.info('memory')['used_memory']
keys_hash = r.dbsize()
print(f"Hash keys:     {keys_hash} keys, {mem_hash:,} bytes")

print(f"\nHashes used {keys_separate - keys_hash} fewer keys!")
print(f"Memory difference: {mem_separate - mem_hash:,} bytes")

---
## 8. Real-World: User Profile CRUD

In [ ]:
r.flushdb()

def create_user(user_id, **fields):
    """Create a new user profile."""
    key = f'user:{user_id}'
    if r.exists(key):
        return False, "User already exists"
    r.hset(key, mapping=fields)
    return True, "User created"

def get_user(user_id):
    """Get a user profile."""
    return r.hgetall(f'user:{user_id}') or None

def update_user(user_id, **fields):
    """Update specific fields of a user."""
    key = f'user:{user_id}'
    if not r.exists(key):
        return False, "User not found"
    r.hset(key, mapping=fields)
    return True, "User updated"

def delete_user(user_id):
    """Delete a user."""
    return r.delete(f'user:{user_id}') > 0

# CREATE
ok, msg = create_user(1001, name='Sujit', email='sujit@example.com', age='25')
print(f"Create: {msg}")

# READ
user = get_user(1001)
print(f"Read: {user}")

# UPDATE
ok, msg = update_user(1001, city='Mumbai', age='26')
print(f"Update: {msg}")
print(f"After update: {get_user(1001)}")

# DELETE
deleted = delete_user(1001)
print(f"Deleted: {deleted}")
print(f"After delete: {get_user(1001)}")

---
## 9. Real-World: Shopping Cart

In [ ]:
def add_to_cart(user_id, product_id, quantity=1):
    """Add a product to the cart (or increase quantity)."""
    r.hincrby(f'cart:{user_id}', product_id, quantity)

def remove_from_cart(user_id, product_id):
    """Remove a product from the cart."""
    r.hdel(f'cart:{user_id}', product_id)

def get_cart(user_id):
    """Get all items in the cart."""
    return r.hgetall(f'cart:{user_id}')

def cart_total_items(user_id):
    """Get total number of items."""
    cart = r.hgetall(f'cart:{user_id}')
    return sum(int(qty) for qty in cart.values())

# Shopping!
add_to_cart('sujit', 'laptop', 1)
add_to_cart('sujit', 'mouse', 2)
add_to_cart('sujit', 'keyboard', 1)
add_to_cart('sujit', 'mouse', 1)  # Buy one more mouse

print("Shopping Cart:")
for product, qty in get_cart('sujit').items():
    print(f"  {product}: {qty}")
print(f"Total items: {cart_total_items('sujit')}")

# Remove keyboard
remove_from_cart('sujit', 'keyboard')
print(f"\nAfter removing keyboard: {get_cart('sujit')}")

---
## 10. Real-World: Configuration Store

In [ ]:
# Store app configuration as a hash
r.hset('config:app', mapping={
    'debug': 'false',
    'max_upload_size': '10485760',  # 10MB in bytes
    'default_language': 'en',
    'maintenance_mode': 'false',
    'items_per_page': '25'
})

# Read individual settings
debug = r.hget('config:app', 'debug')
print(f"Debug mode: {debug}")

# Read all settings at once
config = r.hgetall('config:app')
print(f"\nAll config:")
for key, value in config.items():
    print(f"  {key} = {value}")

# Update a setting
r.hset('config:app', 'maintenance_mode', 'true')
print(f"\nMaintenance mode: {r.hget('config:app', 'maintenance_mode')}")

---
## 11. Real-World: Session Storage with Expiry

In [ ]:
import time

def create_session(session_id, user_data, timeout=1800):  # 30 min default
    """Create a session with auto-expiry."""
    key = f'session:{session_id}'
    r.hset(key, mapping=user_data)
    r.expire(key, timeout)  # Expires the entire hash
    return key

def get_session(session_id):
    """Get session data and refresh timeout."""
    key = f'session:{session_id}'
    data = r.hgetall(key)
    if data:
        r.expire(key, 1800)  # Refresh timeout on access
    return data or None

# Create a session
session = create_session('abc123', {
    'user_id': '1001',
    'username': 'sujit',
    'role': 'admin',
    'login_time': str(time.time())
}, timeout=60)  # 60 seconds for demo

print(f"Session created: {session}")
print(f"Session data: {get_session('abc123')}")
print(f"TTL: {r.ttl('session:abc123')} seconds")

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

```
HSET key field value          → Set a field
HSET key mapping={...}        → Set multiple fields (Python)
HGET key field                → Get a field
HGETALL key                   → Get all fields as dict
HMGET key f1 f2               → Get specific fields
HEXISTS key field             → Check if field exists
HDEL key field                → Delete a field
HLEN key                      → Count fields
HKEYS key / HVALS key         → Get all field names / values
HINCRBY key field amount      → Increment integer field
HINCRBYFLOAT key field amount → Increment float field
HSETNX key field value        → Set only if field doesn't exist
```

### When to Use Hashes
- Representing **objects** (users, products, orders)
- When you need to **read/write individual fields**
- When **memory efficiency** matters (ziplist encoding)
- Shopping carts, config stores, session data

---
## Exercises

1. **Mini User System:** Create 5 user hashes with fields: name, email, age, signup_date. Write functions to: (a) find a user by ID, (b) update email, (c) increment age (birthday), (d) list all field names.

2. **Inventory Tracker:** Create product hashes with fields: name, price, stock. Write functions to: (a) add stock, (b) sell an item (decrement stock, fail if 0), (c) apply a discount (HINCRBYFLOAT with negative value).

3. **Poll System:** Create a hash where fields are poll options and values are vote counts. Write a `vote(poll_id, option)` function and a `results(poll_id)` function that shows percentages.

4. **Hash vs JSON:** Store the same user data as both a hash AND a JSON string. Compare: (a) memory usage, (b) time to read one field, (c) time to read all fields.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 06 — Key Expiry & TTL](./06_Key_Expiry_and_TTL.ipynb)** — Automatic key expiration, session management, and the secret sauce behind Redis caching!